# Hierarchical color clustering

This alternative uses Ward or complete linkage on the label × context count matrix, then evaluates neighbor JSD for each retained K value.


In [62]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial.distance import jensenshannon, squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, leaves_list
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'methods':
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name in ('notebooks', 'color_groupings'):
    PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == 'notebooks' else PROJECT_ROOT.parent.parent.parent
if not (PROJECT_ROOT / 'results').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURES_DIR = PROJECT_ROOT / 'figures' / 'method_comparison'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables' / 'hierarchical_clustering'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

RUN_DATE = '2026-02-25'

## Inputs

Load aggregated counts and county-neighbor context.


In [63]:
counts_path = PROJECT_ROOT / 'results' / 'tables' / 'bayesian_shrinkage' / 'bayesian_shrinkage_aggregated_counts.csv'
if not counts_path.exists():
    counts_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'

df_counts = pd.read_csv(counts_path)
df_counts = df_counts.rename(columns={'fips': 'county', 'lc_type': 'landcover', 'clr': 'color'})
df_counts['county'] = df_counts['county'].astype(str).str.zfill(5)

neighbors_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'ca_county_neighbors.csv'
if not neighbors_path.exists():
    neighbors_path = PROJECT_ROOT / 'dataset' / 'ca_county_neighbors.csv'
neighbors_df = pd.read_csv(neighbors_path)
neighbors_df['county_fips'] = neighbors_df['county_fips'].astype(str).str.zfill(5)
neighbors_df['neighbor_fips'] = neighbors_df['neighbor_fips'].astype(str).str.zfill(5)

neighbors = {}
for _, r in neighbors_df.iterrows():
    c, n = r['county_fips'], r['neighbor_fips']
    neighbors.setdefault(c, set()).add(n)
    neighbors.setdefault(n, set()).add(c)
neighbors = {k: list(v) for k, v in neighbors.items()}

## Context matrix and linkage


In [64]:
counties = sorted(df_counts['county'].unique())
landcovers = sorted(df_counts['landcover'].unique())
colors = sorted(df_counts['color'].unique())

county2i = {c: i for i, c in enumerate(counties)}
lc2j = {l: j for j, l in enumerate(landcovers)}
color2k = {k: idx for idx, k in enumerate(colors)}
k2color = {idx: k for k, idx in color2k.items()}

nC, nL, nK = len(counties), len(landcovers), len(colors)
Y = np.zeros((nC, nL, nK))
for _, r in df_counts.iterrows():
    i = county2i[r['county']]
    j = lc2j[r['landcover']]
    k = color2k[r['color']]
    Y[i, j, k] = r['count']

N = Y.sum(axis=2)

In [65]:
def laplace_smooth(counts, alpha=1.0, K=None):
    counts = np.asarray(counts, dtype=float)
    if K is None:
        K = len(counts)
    n = counts.sum()
    return (counts + alpha) / (n + alpha * K)


def js_dist_matrix(probs, eps=1e-12):
    n = probs.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            p = probs[i] + eps
            q = probs[j] + eps
            p, q = p / p.sum(), q / q.sum()
            D[i, j] = D[j, i] = float(jensenshannon(p, q, base=np.e))
    return D


counts_by_color_lc = Y.sum(axis=0).T
probs = np.array([laplace_smooth(counts_by_color_lc[k]) for k in range(nK)])
dist_matrix = js_dist_matrix(probs)
condensed = squareform(dist_matrix)

In [66]:
Z_ward = linkage(probs, method='ward')
Z_complete = linkage(condensed, method='complete')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dendrogram(Z_ward, labels=colors, ax=axes[0], leaf_rotation=90)
axes[0].set_title('Ward linkage')
dendrogram(Z_complete, labels=colors, ax=axes[1], leaf_rotation=90)
axes[1].set_title('Complete linkage')
plt.tight_layout()
fig.savefig(FIGURES_DIR / f'hierarchical__dendrogram__{RUN_DATE}.png', dpi=200, bbox_inches='tight')
plt.close()

In [67]:
color_totals = df_counts.groupby('color')['count'].sum().to_dict()


def color_to_group_map(cluster_labels, colors):
    groups = {}
    for c, lab in zip(colors, cluster_labels):
        groups.setdefault(lab, []).append(c)
    canonical = {lab: max(members, key=lambda x: (color_totals.get(x, 0), x)) for lab, members in groups.items()}
    return {c: canonical[lab] for c, lab in zip(colors, cluster_labels)}


def evaluate_neighbor_jsd(Y, color_map, neighbors, county2i, lc2j, alpha=1.0, min_n=30):
    group_names = sorted(set(color_map.values()))
    g2idx = {g: i for i, g in enumerate(group_names)}
    nG = len(group_names)
    Yg = np.zeros((nC, nL, nG))
    for k, c in enumerate(colors):
        g = color_map[c]
        j = g2idx[g]
        Yg[:, :, j] += Y[:, :, k]

    edges = set()
    for c_str, nbrs in neighbors.items():
        if c_str not in county2i:
            continue
        for n_str in nbrs:
            if n_str not in county2i:
                continue
            edges.add(tuple(sorted([c_str, n_str])))

    jsd_vals = []
    weights = []
    for (ca, cb) in edges:
        i, j = county2i[ca], county2i[cb]
        for l in range(nL):
            if N[i, l] < min_n or N[j, l] < min_n:
                continue
            p_a = laplace_smooth(Yg[i, l, :], alpha, nG)
            p_b = laplace_smooth(Yg[j, l, :], alpha, nG)
            jsd_vals.append(float(jensenshannon(p_a, p_b, base=np.e)))
            weights.append(min(N[i, l], N[j, l]))
    if not jsd_vals:
        return {'mean_jsd': 0.0, 'max_jsd': 0.0, 'min_jsd': 0.0, 'n_pairs': 0}
    wsum = sum(weights)
    weighted = sum(j * w for j, w in zip(jsd_vals, weights)) / wsum
    return {'mean_jsd': weighted, 'max_jsd': max(jsd_vals), 'min_jsd': min(jsd_vals), 'n_pairs': len(edges)}


def evaluate_surprisal(Y, color_map, neighbors, county2i, alpha=1.0, min_n=30, eps=1e-12):
    """Mean cross-entropy (county vs neighbor-pool): -sum p_county * log(p_pool). Lower = better match."""
    group_names = sorted(set(color_map.values()))
    g2idx = {g: i for i, g in enumerate(group_names)}
    nG = len(group_names)
    Yg = np.zeros((nC, nL, nG))
    for k, c in enumerate(colors):
        g = color_map[c]
        j = g2idx[g]
        Yg[:, :, j] += Y[:, :, k]

    cross_ent_vals = []
    weights = []
    for c_str, nbrs in neighbors.items():
        if c_str not in county2i:
            continue
        i = county2i[c_str]
        nbr_indices = [county2i[n] for n in nbrs if n in county2i]
        for l in range(nL):
            if N[i, l] < min_n:
                continue
            p_county = laplace_smooth(Yg[i, l, :], alpha, nG)
            Y_pool = Yg[i, l, :].copy()
            for j in nbr_indices:
                Y_pool += Yg[j, l, :]
            p_pool = laplace_smooth(Y_pool, alpha, nG) + eps
            ce = -np.sum(p_county * np.log(p_pool))
            cross_ent_vals.append(ce)
            weights.append(N[i, l])
    if not cross_ent_vals:
        return {'mean_surprisal': 0.0}
    wsum = sum(weights)
    weighted = sum(c * w for c, w in zip(cross_ent_vals, weights)) / wsum
    return {'mean_surprisal': float(weighted)}

## Scoring alternatives

Evaluate each retained linkage method and K value with neighbor JSD and surprisal.


In [68]:
K_values = [4, 6, 8, 10, 12]
results = []

for method_name, Z in [('ward', Z_ward), ('complete', Z_complete)]:
    for K in K_values:
        labels = fcluster(Z, K, criterion='maxclust')
        color_map = color_to_group_map(labels, colors)
        stats = evaluate_neighbor_jsd(Y, color_map, neighbors, county2i, lc2j)
        stats.update(evaluate_surprisal(Y, color_map, neighbors, county2i))
        stats['method'] = method_name
        stats['K'] = K
        results.append(stats)

df_results = pd.DataFrame(results)

In [69]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for method_name in ['ward', 'complete']:
    sub = df_results[df_results['method'] == method_name]
    axes[0].plot(sub['K'], sub['mean_jsd'], 'o-', label=method_name)
    axes[1].plot(sub['K'], sub['mean_surprisal'], 'o-', label=method_name)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Mean weighted neighbor JSD')
axes[0].legend()
axes[0].set_xticks(K_values)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Mean surprisal (cross-entropy county vs pool)')
axes[1].legend()
axes[1].set_xticks(K_values)
plt.tight_layout()
fig.savefig(FIGURES_DIR / f'hierarchical__jsd_vs_k__{RUN_DATE}.png', dpi=200, bbox_inches='tight')
plt.close()

## Selection summary and exports


In [70]:
best_jsd_row = df_results.loc[df_results['mean_jsd'].idxmin()]
best_surprisal_row = df_results.loc[df_results['mean_surprisal'].idxmin()]

def get_groups(color_map):
    groups_by_canon = {}
    for c, g in color_map.items():
        groups_by_canon.setdefault(g, []).append(c)
    return groups_by_canon

def print_groups(color_map, label):
    groups_by_canon = get_groups(color_map)
    print(f"--- Best by {label} ---")
    for canon in sorted(groups_by_canon, key=lambda x: -len(groups_by_canon[x])):
        print(f"  {canon} <- {sorted(groups_by_canon[canon])}")

Z_jsd = Z_ward if best_jsd_row['method'] == 'ward' else Z_complete
Z_surprisal = Z_ward if best_surprisal_row['method'] == 'ward' else Z_complete
labels_jsd = fcluster(Z_jsd, int(best_jsd_row['K']), criterion='maxclust')
labels_surprisal = fcluster(Z_surprisal, int(best_surprisal_row['K']), criterion='maxclust')
final_map_jsd = color_to_group_map(labels_jsd, colors)
final_map_surprisal = color_to_group_map(labels_surprisal, colors)

print_groups(final_map_jsd, f"JSD (method={best_jsd_row['method']}, K={int(best_jsd_row['K'])}, mean_jsd={best_jsd_row['mean_jsd']:.4f})")
print()
print_groups(final_map_surprisal, f"surprisal (method={best_surprisal_row['method']}, K={int(best_surprisal_row['K'])}, mean_surprisal={best_surprisal_row['mean_surprisal']:.4f})")

def groups_to_df(color_map):
    out = []
    for canon in set(color_map.values()):
        for m in [c for c in color_map if color_map[c] == canon]:
            out.append({'color': m, 'group': canon})
    return pd.DataFrame(out)

pd.DataFrame(groups_to_df(final_map_jsd)).to_csv(TABLES_DIR / f'hierarchical__color_groups_best_jsd__{RUN_DATE}.csv', index=False)
pd.DataFrame(groups_to_df(final_map_surprisal)).to_csv(TABLES_DIR / f'hierarchical__color_groups_best_surprisal__{RUN_DATE}.csv', index=False)
df_results.to_csv(TABLES_DIR / f'hierarchical__jsd_by_k__{RUN_DATE}.csv', index=False)

--- Best by JSD (method=complete, K=4, mean_jsd=0.1709) ---
  cocoa <- ['amber', 'auburn', 'azure', 'beige', 'cocoa', 'foo', 'gold', 'ivory', 'lemon', 'lilac', 'maroon', 'navy', 'orange', 'purple', 'red', 'scarlet', 'sienna', 'terracotta']
  brown <- ['alabaster', 'bar', 'blue', 'brown', 'coffee', 'green', 'grey', 'indigo', 'lavender', 'olive', 'plum', 'sage']
  gray <- ['aqua', 'aquamarine', 'crimson', 'gray', 'yellow']
  verde <- ['emerald', 'tan', 'verde']

--- Best by surprisal (method=complete, K=4, mean_surprisal=0.5475) ---
  cocoa <- ['amber', 'auburn', 'azure', 'beige', 'cocoa', 'foo', 'gold', 'ivory', 'lemon', 'lilac', 'maroon', 'navy', 'orange', 'purple', 'red', 'scarlet', 'sienna', 'terracotta']
  brown <- ['alabaster', 'bar', 'blue', 'brown', 'coffee', 'green', 'grey', 'indigo', 'lavender', 'olive', 'plum', 'sage']
  gray <- ['aqua', 'aquamarine', 'crimson', 'gray', 'yellow']
  verde <- ['emerald', 'tan', 'verde']


## Context heatmap


In [71]:
leaf_order = leaves_list(Z_jsd)
probs_ordered = probs[leaf_order]
colors_ordered = [colors[i] for i in leaf_order]

df_heat = pd.DataFrame(probs_ordered, index=colors_ordered, columns=landcovers)
fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(df_heat, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'P(landcover|color)'})
ax.set_title(f'Color × landcover (rows ordered by {best_jsd_row["method"]} dendrogram)')
fig.savefig(FIGURES_DIR / f'hierarchical__heatmap_color_context__{RUN_DATE}.png', dpi=200, bbox_inches='tight')
plt.close()

In [72]:
summary_jsd = {
    'criterion': 'best_jsd',
    'method': best_jsd_row['method'],
    'K': int(best_jsd_row['K']),
    'mean_jsd': float(best_jsd_row['mean_jsd']),
    'mean_surprisal': float(best_jsd_row['mean_surprisal']),
    'n_raw_colors': nK,
}
summary_surprisal = {
    'criterion': 'best_surprisal',
    'method': best_surprisal_row['method'],
    'K': int(best_surprisal_row['K']),
    'mean_jsd': float(best_surprisal_row['mean_jsd']),
    'mean_surprisal': float(best_surprisal_row['mean_surprisal']),
    'n_raw_colors': nK,
}
pd.DataFrame([summary_jsd, summary_surprisal])

,criterion,method,K,mean_jsd,mean_surprisal,n_raw_colors
0,best_jsd,complete,4,0.170942,0.547521,38
1,best_surprisal,complete,4,0.170942,0.547521,38


## Optional follow-up

Compare the selected hierarchy with the greedy pooling alternative using the same export schemas.
